# Notebook 6: Train Models
## Car Price Prediction Project

**Objective:** Train three manual models:
1. **Linear Regression**
2. **Decision Tree Regressor**
3. **XGBoost Regressor**

---

## 6.1 Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install xgboost -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

## 6.2 Load Preprocessed Data

In [ ]:
# Scaled data for Linear Regression
X_train_scaled = pd.read_csv('/content/drive/MyDrive/X_train_scaled.csv')
X_test_scaled = pd.read_csv('/content/drive/MyDrive/X_test_scaled.csv')

# Unscaled data for tree-based models
X_train = pd.read_csv('/content/drive/MyDrive/X_train.csv')
X_test = pd.read_csv('/content/drive/MyDrive/X_test.csv')

# Target
y_train = pd.read_csv('/content/drive/MyDrive/y_train.csv').squeeze()
y_test = pd.read_csv('/content/drive/MyDrive/y_test.csv').squeeze()

print(f"Train: {X_train.shape[0]} samples, {X_train.shape[1]} features")
print(f"Test:  {X_test.shape[0]} samples")

## 6.3 Helper Function

In [ ]:
def evaluate_model(name, model, X_tr, X_te, y_tr, y_te):
    """Train, predict, and evaluate a model."""
    # Train
    model.fit(X_tr, y_tr)

    # Predict
    y_train_pred = model.predict(X_tr)
    y_test_pred = model.predict(X_te)

    # Metrics
    results = {
        'Model': name,
        'Train R²': r2_score(y_tr, y_train_pred),
        'Test R²': r2_score(y_te, y_test_pred),
        'Train RMSE': np.sqrt(mean_squared_error(y_tr, y_train_pred)),
        'Test RMSE': np.sqrt(mean_squared_error(y_te, y_test_pred)),
        'Train MAE': mean_absolute_error(y_tr, y_train_pred),
        'Test MAE': mean_absolute_error(y_te, y_test_pred)
    }

    print(f"\n{'='*60}")
    print(f"  {name}")
    print(f"{'='*60}")
    print(f"  Train R²:  {results['Train R²']:.4f}    |  Test R²:  {results['Test R²']:.4f}")
    print(f"  Train RMSE: ${results['Train RMSE']:,.0f}  |  Test RMSE: ${results['Test RMSE']:,.0f}")
    print(f"  Train MAE:  ${results['Train MAE']:,.0f}  |  Test MAE:  ${results['Test MAE']:,.0f}")

    return results, y_test_pred

## 6.4 Model 1: Linear Regression

Uses **scaled** data because linear regression is sensitive to feature magnitudes.

In [ ]:
lr_model = LinearRegression()
lr_results, lr_preds = evaluate_model(
    'Linear Regression', lr_model,
    X_train_scaled, X_test_scaled, y_train, y_test
)

In [ ]:
# Linear Regression Coefficients
coef_df = pd.DataFrame({
    'Feature': X_train_scaled.columns,
    'Coefficient': lr_model.coef_
}).sort_values('Coefficient', key=abs, ascending=False)

print(f"Intercept: ${lr_model.intercept_:,.2f}")
print(f"\nTop 15 coefficients (by absolute value):")
coef_df.head(15)

In [ ]:
# Visualize top coefficients
top_coef = coef_df.head(15)
colors = ['green' if c > 0 else 'red' for c in top_coef['Coefficient']]

plt.figure(figsize=(10, 8))
plt.barh(range(len(top_coef)), top_coef['Coefficient'], color=colors)
plt.yticks(range(len(top_coef)), top_coef['Feature'])
plt.xlabel('Coefficient Value')
plt.title('Linear Regression — Top 15 Coefficients')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 6.5 Model 2: Decision Tree Regressor

Uses **unscaled** data — decision trees are invariant to feature scaling.

In [ ]:
dt_model = DecisionTreeRegressor(max_depth=5, random_state=42)
dt_results, dt_preds = evaluate_model(
    'Decision Tree', dt_model,
    X_train, X_test, y_train, y_test
)

In [ ]:
# Feature importance from Decision Tree
dt_importance = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': dt_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("Decision Tree — Top 15 Feature Importances:")
dt_importance.head(15)

In [ ]:
# Visualize the Decision Tree
from sklearn.tree import plot_tree

plt.figure(figsize=(24, 12))
plot_tree(dt_model, feature_names=X_train.columns, filled=True,
          rounded=True, fontsize=8, max_depth=3)
plt.title('Decision Tree (showing top 3 levels)', fontsize=16)
plt.tight_layout()
plt.show()

## 6.6 Model 3: XGBoost Regressor

Uses **unscaled** data — XGBoost is tree-based and doesn't require scaling.

In [ ]:
xgb_model = XGBRegressor(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    random_state=42,
    verbosity=0
)
xgb_results, xgb_preds = evaluate_model(
    'XGBoost', xgb_model,
    X_train, X_test, y_train, y_test
)

In [ ]:
# Feature importance from XGBoost
xgb_importance = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': xgb_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("XGBoost — Top 15 Feature Importances:")
xgb_importance.head(15)

In [ ]:
# Visualize XGBoost feature importance
top_xgb = xgb_importance.head(15)

plt.figure(figsize=(10, 8))
plt.barh(range(len(top_xgb)), top_xgb['Importance'], color='steelblue')
plt.yticks(range(len(top_xgb)), top_xgb['Feature'])
plt.xlabel('Importance')
plt.title('XGBoost — Top 15 Feature Importances')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 6.7 Model Comparison

In [ ]:
# Summary table
results_df = pd.DataFrame([lr_results, dt_results, xgb_results])
results_df = results_df.set_index('Model')

print("\n" + "="*70)
print("  MODEL COMPARISON")
print("="*70)
results_df

In [ ]:
# Visual comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
models = ['Linear Regression', 'Decision Tree', 'XGBoost']
colors = ['#e74c3c', '#2ecc71', '#3498db']

# R² comparison
test_r2 = [lr_results['Test R²'], dt_results['Test R²'], xgb_results['Test R²']]
axes[0].bar(models, test_r2, color=colors, edgecolor='white')
axes[0].set_title('Test R²')
axes[0].set_ylim(0, 1)
for i, v in enumerate(test_r2):
    axes[0].text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold')

# RMSE comparison
test_rmse = [lr_results['Test RMSE'], dt_results['Test RMSE'], xgb_results['Test RMSE']]
axes[1].bar(models, test_rmse, color=colors, edgecolor='white')
axes[1].set_title('Test RMSE ($)')
for i, v in enumerate(test_rmse):
    axes[1].text(i, v + 100, f'${v:,.0f}', ha='center', fontweight='bold')

# MAE comparison
test_mae = [lr_results['Test MAE'], dt_results['Test MAE'], xgb_results['Test MAE']]
axes[2].bar(models, test_mae, color=colors, edgecolor='white')
axes[2].set_title('Test MAE ($)')
for i, v in enumerate(test_mae):
    axes[2].text(i, v + 100, f'${v:,.0f}', ha='center', fontweight='bold')

plt.suptitle('Model Performance Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6.8 Actual vs Predicted Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
preds_list = [lr_preds, dt_preds, xgb_preds]

for ax, name, preds, color in zip(axes, models, preds_list, colors):
    ax.scatter(y_test, preds, alpha=0.6, color=color, s=40)
    ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
            'k--', linewidth=1.5, label='Perfect prediction')
    ax.set_xlabel('Actual Price ($)')
    ax.set_ylabel('Predicted Price ($)')
    ax.set_title(f'{name}')
    ax.legend()

plt.suptitle('Actual vs Predicted Price', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6.9 Save Models and Results

In [ ]:
import pickle

# Save models
models_dict = {
    'linear_regression': lr_model,
    'decision_tree': dt_model,
    'xgboost': xgb_model
}

for name, model in models_dict.items():
    with open(f'/content/drive/MyDrive/{name}_model.pkl', 'wb') as f:
        pickle.dump(model, f)

# Save results
results_df.to_csv('/content/drive/MyDrive/model_results.csv')

print("Models and results saved to Google Drive.")

---
## Training Summary

| Model | Test R² | Test RMSE | Test MAE |
|-------|---------|-----------|----------|
| Linear Regression | See output | See output | See output |
| Decision Tree | See output | See output | See output |
| XGBoost | See output | See output | See output |

**Key observations:**
- Linear Regression provides a baseline with interpretable coefficients
- Decision Tree captures non-linear relationships and is visually interpretable
- XGBoost typically achieves the best performance through boosting

**Next step →** Notebook 07: Evaluation